# Unix 3C: Slurm mapping challenge

This notebook is the independent challenge for Unix session 3. You have already inspected the inputs, run the mapping workflow locally, validated the BAM, and interpreted coverage. Your task now is to move that same workflow to Ada as a scheduled Slurm job.

## Learning outcomes

By the end you should be able to adapt a local bioinformatics workflow into a Slurm script, request appropriate resources, submit and monitor the job, inspect logs, and compare the cluster output with the local run.


## 1. Challenge brief

Create and submit a Slurm job that maps the `barcode09/` reads to `reference/HVol_Complete.fasta`, produces a sorted and indexed BAM, validates the result, and writes coverage summaries.

The scientific workflow should stay the same as the local version:

```text
FASTQ reads + FASTA reference
        |
        v
minimap2 -ax map-ont
        |
        v
samtools sort
        |
        v
sorted BAM + BAI
        |
        v
quickcheck, flagstat, idxstats, depth -aa
```

Your job is to package that workflow so it runs correctly under the scheduler.


## 2. Prepare the course environment on Ada

Before writing the job script, make sure the course repository and conda environment are available on Ada.

```bash
mkdir -p ~/Documents/LIFE4138
cd ~/Documents/LIFE4138
git clone git@github.com:BioinformaticsMSc/LIFE4138_2627.git
cd LIFE4138_2627
```

Update your course checkout:

```bash
cd ~/Documents/LIFE4138/LIFE4138_2627
# If your repository is somewhere else, cd there instead.
git pull
```

If the `lectures` environment does not exist yet, create it from the repository environment file:

```bash
# Load the currently supported Conda/Mamba module first if Ada requires one.
conda env create -f lectures.yml
conda activate lectures
```

If the environment already exists, update it:

```bash
# Load the currently supported Conda/Mamba module first if Ada requires one.
conda env update -f lectures.yml --prune
conda activate lectures
```

Check that the required tools are available:

```bash
which minimap2
minimap2 --version
which samtools
samtools --version
```

Your Slurm script must also activate the same environment. A batch job may not inherit the conda setup from your interactive shell, so include the current Ada-supported Conda/Mamba initialisation before `conda activate lectures`.


OR!!! Use the appropriate modules on ADA.


## 3. Write your own Slurm script

Create your own script from a blank file. Do not copy a completed template.

Name it `work/my_align_barcode09.slurm`. It should be possible for another person to read the script and understand what resources were requested, how the software was loaded, which inputs were used, where outputs were written, and how the result was checked.


In [ ]:
%%bash
mkdir -p work logs
: > work/my_align_barcode09.slurm
ls -lh work/my_align_barcode09.slurm


## 4. Script requirements

Your script must include the following pieces, written by you:

- a shebang line;
- Slurm directives for job name, stdout log, stderr log, time, memory, and CPUs;
- current Ada-supported Conda/Mamba setup and `conda activate lectures`;
- commands that create an output directory under `work/`;
- the Minimap2 to Samtools sorting pipeline;
- BAM indexing;
- `samtools quickcheck`;
- `samtools flagstat`;
- `samtools idxstats`;
- `samtools depth -aa`;
- a small coverage summary;
- software version output.



### Minimal structure, not a template

Your script should have this shape, but you need to fill in the real directives, software setup, paths, and commands yourself:

```bash
#!/usr/bin/env bash

#SBATCH ...
#SBATCH ...

set -euo pipefail

# Initialise Conda/Mamba if needed, then activate the lectures environment.

# Define input paths and an output directory.

# Run minimap2 and stream into samtools sort.

# Index and validate the BAM.

# Produce mapping and coverage summaries.

# Record software versions.
```

If your script is just this skeleton with missing details, it is not complete.


## 5. Submit the job on Ada

Submit from the session directory on Ada, after checking that the inputs are present there and that your script is not empty.

```bash
bash -n work/my_align_barcode09.slurm
sbatch work/my_align_barcode09.slurm
```

`bash -n` checks the shell syntax before submitting. It does not prove that the job is scientifically correct, but it catches many simple script mistakes.

Record the job ID returned by `sbatch`. That ID connects the submission, logs, accounting information, and output files.


## 6. Monitor and inspect

While the job is waiting or running, check its queue state:

```bash
squeue -u "$USER"
```

After the job finishes, inspect accounting information:

```bash
sacct -j JOB_ID --format=JobID,JobName,State,Elapsed,AllocCPUS,MaxRSS,ExitCode
```

Then read the stdout and stderr logs. A completed job is not automatically a scientifically correct job.


## 7. Validate the outputs

Your output directory should contain a sorted BAM, its BAI index, and text summaries. Use commands like these, adapted to your output path:

```bash
samtools quickcheck -v work/JOB_OUTPUT/aligned_barcode09.sorted.bam
samtools flagstat work/JOB_OUTPUT/aligned_barcode09.sorted.bam | head
samtools idxstats work/JOB_OUTPUT/aligned_barcode09.sorted.bam
samtools depth -aa work/JOB_OUTPUT/aligned_barcode09.sorted.bam > work/JOB_OUTPUT/coverage_barcode09.tsv
```

Compare the Slurm results with the local notebook results. Differences should be explainable by paths, versions, settings, or input changes. Silent differences are a reason to stop and investigate.


## 8. Thread-count experiment

Once you have one correct run, try a small resource experiment. Change only the resource and thread settings, not the scientific inputs.

Complete `thread_experiment_results.tsv` with:

- job ID;
- requested CPUs;
- Minimap2 threads;
- Samtools sort threads;
- elapsed time;
- peak memory;
- output validation result;
- one checksum or alignment count that confirms scientific equivalence.

Separate queue wait from execution time. More requested CPUs will not necessarily make this small dataset faster.


In [ ]:
%%bash
cat thread_experiment_results.tsv


## 9. Evidence to submit

Submit or retain:

- your own Slurm script;
- the job ID;
- stdout and stderr logs;
- `sacct` output;
- sorted BAM and BAI if requested by the instructor;
- `flagstat`, `idxstats`, and coverage summaries;
- software versions;
- completed thread-count table;
- a short note comparing the Slurm result with the local run.


## Exit ticket

Before leaving, answer these in your notes.

1. Which parts of the local workflow changed when you moved to Slurm?
2. Which parts should not have changed?
3. How did you decide how many CPUs to request?
4. What evidence shows the job ran successfully?
5. What evidence shows the result is scientifically comparable with the local run?
